In [ ]:
# Example 1: open an in-memory Database and inspect the public API surface.
from apswutils import Database

db = Database(":memory:")
print(f"Database type: {type(db).__name__}")
print(f"db.conn type: {type(db.conn).__name__}")

# Tables collection exists and is initially empty
tables = db.tables
print(f"Initial tables: {tables}")
assert isinstance(tables, list)
assert tables == []


In [ ]:
# Example 2: create a table by inserting a row, then read it back.
# Tables are addressed via db["name"] — the `.t` accessor is fastlite sugar
# layered on top of apswutils, not part of apswutils itself.
from apswutils import Database

db = Database(":memory:")
people = db["people"]
people.insert({"id": 1, "name": "Alice", "age": 30}, pk="id")
people.insert({"id": 2, "name": "Bob", "age": 25}, pk="id")

rows = list(people.rows)
print(f"Rows: {rows}")
assert len(rows) == 2
assert rows[0]["name"] == "Alice"
assert people.count == 2

In [ ]:
# Example 3: execute raw SQL via the Database's execute / query helpers.
from apswutils import Database

db = Database(":memory:")
db.execute("CREATE TABLE nums (n INTEGER)")
for i in range(5):
    db.execute("INSERT INTO nums (n) VALUES (?)", [i])

results = list(db.query("SELECT n FROM nums ORDER BY n"))
print(f"Query results: {results}")
assert len(results) == 5
assert results[0]["n"] == 0
assert results[-1]["n"] == 4


In [ ]:
# Example 4: schema introspection - columns, pks, indexes, schema string.
from apswutils import Database

db = Database(":memory:")
items = db["items"]
items.insert_all([
    {"id": 1, "label": "a", "weight": 1.5},
    {"id": 2, "label": "b", "weight": 2.5},
], pk="id")

col_names = [c.name for c in items.columns]
print(f"Columns: {col_names}")
print(f"Primary keys: {items.pks}")
print(f"Schema:\n{items.schema}")
assert "id" in col_names
assert "label" in col_names
assert items.pks == ["id"]

In [ ]:
# Example 5: update + delete row-level operations on a Table.
from apswutils import Database

db = Database(":memory:")
books = db["books"]
books.insert_all([
    {"id": 1, "title": "Book One", "pages": 100},
    {"id": 2, "title": "Book Two", "pages": 200},
    {"id": 3, "title": "Book Three", "pages": 300},
], pk="id")

books.update(2, {"pages": 250})
books.delete(3)

remaining = sorted(list(books.rows), key=lambda r: r["id"])
print(f"Remaining rows: {remaining}")
assert books.count == 2
assert remaining[1]["pages"] == 250
assert all(r["id"] != 3 for r in remaining)
print("All assertions passed")